# Lesson 10 — UART 是字节流，不是消息队列

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


串口一次读到的内容可能只有半帧，也可能有多帧。把每一次 read 当成一条完整消息会让协议在真实通信中失效。

## 最小知识

本课用换行结束帧：`PING\n` 返回 `PONG\n`，`LED:1\n` / `LED:0\n` 返回 `OK\n`。`input.text` 是这次到达的字节，Controller 必须积累到换行才解析。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 10-uart-terminal guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){if(b_.input.text=="PING\n")b_.output.outgoing+="PONG\n";}private:Device&b_;};}


## 2. 修改一次

增加 `STATUS\n`，返回 LED 状态。先定义精确字符串，再写测试。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 10-uart-terminal modify
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){if(b_.input.text=="PING\n")b_.output.outgoing+="PONG\n";}private:Device&b_;};}


## 3. 故障实验（Debug）

下面代码把这一次收到的 chunk 直接判断，分两段到达的 `PI` + `NG\n` 无法识别。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 10-uart-terminal broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){if(b_.input.text=="PING\n")b_.output.outgoing+="PONG\n";}private:Device&b_;};}


## 4. 你来完成（Build）

维护接收缓冲区，逐帧解析；断连时设置 error。未知帧输出 `ERR\n`。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/10-uart-terminal/hints.md)。


In [ ]:
%%academy_lab 10-uart-terminal exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){// TODO: 累积字节到 \n，再解析 PING / LED
}private:Device&b_;};}


## 5. 开放挑战

给帧加入长度上限和 CRC。遇到超长或校验失败时，怎样重新同步到下一帧？

## 你刚刚真正学到了什么？

UART 只保证字节顺序，不保证你的消息边界。帧化、缓冲和错误恢复属于应用协议。

## 硬件迁移

STEP 2 用 UART driver 的 read/write；波特率、电平、缓冲区溢出和实际设备超时需加入 Hardware Checklist。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/10-uart-terminal-solution.ipynb)。
